In [1]:
import csv, json, random, re, shutil
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


# Perpare for Work

In [2]:
tts = TTS(model_name="tts_models/multilingual/multi-dataset/your_tts").to(device)

In [3]:
print("Supported languages:", tts.languages)
print("Lithuanian supported:", any(l.lower() in ("lt", "lt-lt", "lit", "lithuanian") for l in tts.languages))

Supported languages: ['en', 'fr-fr', 'pt-br']
Lithuanian supported: False


# Zero-Shot Speech Synthesis (English)

In [4]:
ANALYSIS_SR = 16000

def audio_info(path):
    info = sf.info(path)
    return {
        "file": str(path),
        "duration_s": round(info.duration, 3),
        "sampling_rate_hz": info.samplerate,
        "channels": info.channels,
        "format": info.format,      
        "subtype": info.subtype,    
    }


def f0_contour(y, sr):
    f0, voiced, _ = librosa.pyin(y, fmin=60, fmax=500, sr=sr, frame_length=1024)
    t = librosa.times_like(f0, sr=sr, hop_length=1024 // 4)
    return t, f0


def mag_spectrum_db(y, sr, n_fft=1024, hop=256):
    """Average magnitude spectrum in dB (Hann-windowed frames)."""
    if len(y) < n_fft:
        y = np.pad(y, (0, n_fft - len(y)))
    win = np.hanning(n_fft)
    frames = np.array([y[i:i + n_fft] * win for i in range(0, len(y) - n_fft + 1, hop)])
    mag = np.abs(np.fft.rfft(frames, axis=1)).mean(axis=0)
    return np.fft.rfftfreq(n_fft, 1 / sr), 20 * np.log10(mag + 1e-10)


def compare_plots(nat_path, syn_path, title):
    y_nat, _ = librosa.load(nat_path, sr=ANALYSIS_SR, mono=True)
    y_syn, _ = librosa.load(syn_path, sr=ANALYSIS_SR, mono=True)
    fig, ax = plt.subplots(3, 2, figsize=(13, 9))
    for col, (name, y) in enumerate([("Natural", y_nat), ("Synthesized", y_syn)]):
        t = np.arange(len(y)) / ANALYSIS_SR
        ax[0, col].plot(t, y, lw=0.6)
        ax[0, col].set(title=f"{name}: waveform ({len(y)/ANALYSIS_SR:.2f} s)", xlabel="Time, s", ylabel="Amplitude")

        fr, db = mag_spectrum_db(y, ANALYSIS_SR)
        ax[1, col].plot(fr, db, lw=0.8)
        ax[1, col].set(title=f"{name}: magnitude spectrum", xlabel="Frequency, Hz", ylabel="Magnitude, dB")

        tf, f0 = f0_contour(y, ANALYSIS_SR)
        ax[2, col].plot(tf, f0, ".", ms=3)
        ax[2, col].set(title=f"{name}: F0 contour", xlabel="Time, s", ylabel="F0, Hz", ylim=(50, 500))

        v = f0[~np.isnan(f0)]
        if len(v):
            print(f"[{name}] duration={len(y)/ANALYSIS_SR:.2f}s  F0 mean={v.mean():.1f} Hz  min={v.min():.1f}  "
                  f"max={v.max():.1f}  std={v.std():.1f}  voiced={100*len(v)/len(f0):.0f}%")
        else:
            print(f"[{name}] no voiced frames detected")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    return fig

In [5]:
import sounddevice as sd

def record(path, seconds=5, sr=16000):
    print(f"Recording {seconds} s ... speak now")
    audio = sd.rec(int(seconds * sr), samplerate=sr, channels=1, dtype="int16")
    sd.wait()
    sf.write(path, audio, sr, subtype="PCM_16")
    print("saved", path)

In [ ]:
# This is a test of the speech synthesis system.
# record("reference_en.wav", seconds=5)

Recording 5 s ... speak now
saved reference_en.wav


In [6]:
print(json.dumps(audio_info("reference_en.wav"), indent=2))

{
  "file": "reference_en.wav",
  "duration_s": 5.0,
  "sampling_rate_hz": 16000,
  "channels": 1,
  "format": "WAV",
  "subtype": "PCM_16"
}


In [7]:
text_en = "This is a test of the speech synthesis system."
tts.tts_to_file(
    text=text_en,
    speaker_wav="reference_en.wav",
    language="en",
    file_path="output.wav",
)
print(json.dumps(audio_info("output.wav"), indent=2))

{
  "file": "output.wav",
  "duration_s": 3.777,
  "sampling_rate_hz": 16000,
  "channels": 1,
  "format": "WAV",
  "subtype": "PCM_16"
}


In [8]:
from IPython.display import Audio, display
print("Natural:");     display(Audio("reference_en.wav"))
print("Synthesized:"); display(Audio("output.wav"))

Natural:


Synthesized:


In [26]:
fig = compare_plots("reference_en.wav", "output.wav", "Natural vs Synthesized (English)")
fig.savefig("comparison_en.png", dpi=150)

[Natural] duration=5.00s  F0 mean=98.9 Hz  min=60.0  max=151.2  std=23.8  voiced=69%
[Synthesized] duration=4.16s  F0 mean=119.3 Hz  min=100.3  max=137.0  std=8.8  voiced=59%


C:\Users\avark\AppData\Local\Temp\ipykernel_17624\512406003.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Zero-Shot Speech Synthesis (Lithuanian)

In [23]:
# Tai yra šnekos sintezės sistemos testas.
record("reference_lt.wav", seconds=5) 
print(json.dumps(audio_info("reference_lt.wav"), indent=2))

Recording 5 s ... speak now
saved reference_lt.wav
{
  "file": "reference_lt.wav",
  "duration_s": 5.0,
  "sampling_rate_hz": 16000,
  "channels": 1,
  "format": "WAV",
  "subtype": "PCM_16"
}


In [ ]:
text_lt = "Tai yra šnekos sintezės sistemos testas."
tts.tts_to_file(
    text=text_lt,
    speaker_wav="reference_lt.wav",
    language="en",
    file_path="output_lt.wav",
)
print(json.dumps(audio_info("output_lt.wav"), indent=2))
print("Natural:");     display(Audio("reference_lt.wav"))
print("Synthesized:"); display(Audio("output_lt.wav"))

{
  "file": "output_lt.wav",
  "duration_s": 3.729,
  "sampling_rate_hz": 16000,
  "channels": 1,
  "format": "WAV",
  "subtype": "PCM_16"
}
Natural:


Synthesized:


In [28]:
fig = compare_plots("reference_lt.wav", "output_lt.wav", "Natural vs Synthesized (Lithuanian)")
fig.savefig("comparison_lt.png", dpi=150)

[Natural] duration=5.00s  F0 mean=96.5 Hz  min=60.0  max=146.0  std=24.5  voiced=81%
[Synthesized] duration=3.73s  F0 mean=116.9 Hz  min=94.2  max=137.0  std=8.0  voiced=53%


C:\Users\avark\AppData\Local\Temp\ipykernel_17624\512406003.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Lithuanian Dataset Preparation

In [ ]:
import struct, zlib
from pathlib import Path, PurePosixPath

def _name(raw, flags):
    return raw.decode("utf-8") if flags & 0x800 else raw.decode("cp437")

def find_central_directory(data):
    tail_off = max(0, len(data) - 4 * 1024 * 1024)
    tail = data[tail_off:]
    z64 = tail.rfind(b"PK\x06\x06")
    eocd = tail.rfind(b"PK\x05\x06")
    if z64 != -1:
        cdsize = struct.unpack_from("<Q", tail, z64 + 40)[0]; end = tail_off + z64
    elif eocd != -1:
        cdsize = struct.unpack_from("<I", tail, eocd + 12)[0]; end = tail_off + eocd
    else:
        raise ValueError("no end-of-central-directory record found in this part")
    start = end - cdsize
    if start < 0 or data[start:start + 4] != b"PK\x01\x02":
        raise ValueError("central directory does not start inside this part")
    return start, end

def parse_central_directory(data, start, end):
    entries, pos = {}, start
    while pos + 46 <= end and data[pos:pos + 4] == b"PK\x01\x02":
        (_, vm, vn, flags, method, mt, md, crc, csize, usize,
         nlen, elen, clen, disk, ia, ea, lho) = struct.unpack_from("<4s6H3I5H2I", data, pos)
        name = _name(data[pos + 46: pos + 46 + nlen], flags)
        extra = data[pos + 46 + nlen: pos + 46 + nlen + elen]
        j = 0
        while j + 4 <= len(extra):  
            hid, sz = struct.unpack_from("<HH", extra, j)
            if hid == 1:
                k = j + 4
                if usize == 0xFFFFFFFF: usize = struct.unpack_from("<Q", extra, k)[0]; k += 8
                if csize == 0xFFFFFFFF: csize = struct.unpack_from("<Q", extra, k)[0]; k += 8
                if lho == 0xFFFFFFFF:   lho = struct.unpack_from("<Q", extra, k)[0]; k += 8
            j += 4 + sz
        entries[name] = dict(method=method, crc=crc, csize=csize, usize=usize, flags=flags)
        pos += 46 + nlen + elen + clen
    return entries

def safe_target(out_dir, name):
    p = PurePosixPath(name.replace("\\", "/"))
    if p.is_absolute() or ".." in p.parts or not p.parts:
        return None
    return Path(out_dir).joinpath(*p.parts)

def extract_local_files(data, entries, out_dir, limit_end):
    ok = bad = partial = 0
    i = data.find(b"PK\x03\x04")
    while i != -1:
        if i + 30 <= len(data):
            _, ver, flags, method, mt, md, crc, csize, usize, nlen, elen = struct.unpack_from("<4s5H3I2H", data, i)
            name = _name(data[i + 30: i + 30 + nlen], flags)
            e = entries.get(name)
            if e and not name.endswith("/"):
                ds = i + 30 + nlen + elen
                de = ds + e["csize"]
                if de > limit_end:
                    partial += 1
                else:
                    raw = data[ds:de]
                    try:
                        content = raw if e["method"] == 0 else zlib.decompress(raw, -15)
                        if zlib.crc32(content) & 0xFFFFFFFF == e["crc"]:
                            target = safe_target(out_dir, name)
                            if target:
                                target.parent.mkdir(parents=True, exist_ok=True)
                                target.write_bytes(content); ok += 1
                        else:
                            bad += 1
                    except Exception:
                        bad += 1
        i = data.find(b"PK\x03\x04", i + 4)
    return ok, bad, partial   

In [ ]:
from collections import Counter
ARCHIVE = Path("emo-wavs.zip.021")
data = ARCHIVE.read_bytes()
cd_start, cd_end = find_central_directory(data)
entries = parse_central_directory(data, cd_start, cd_end)

In [45]:
SRC = Path("emo_extracted"); SRC.mkdir(exist_ok=True)
ok, bad, partial = extract_local_files(data, entries, SRC, cd_start)
print(f"extracted: {ok}, checksum errors: {bad}, cut-off files skipped: {partial}")
wavs = sorted(SRC.rglob("*.wav"))
print(len(wavs), "wav files; first 10:")
for w in wavs[:10]:
    print("  ", w.relative_to(SRC))

extracted: 437, checksum errors: 0, cut-off files skipped: 0
437 wav files; first 10:
   wavs\Z\M4\4810-M4Z\4810-M4Z-0215.wav
   wavs\Z\M4\4810-M4Z\4810-M4Z-0216.wav
   wavs\Z\M4\4810-M4Z\4810-M4Z-0217.wav
   wavs\Z\M4\4810-M4Z\4810-M4Z-0218.wav
   wavs\Z\M4\4810-M4Z\4810-M4Z-0219.wav
   wavs\Z\M4\4810-M4Z\4810-M4Z-0220.wav
   wavs\Z\M4\4810-M4Z\4810-M4Z-0221.wav
   wavs\Z\M4\4810-M4Z\4810-M4Z-0222.wav
   wavs\Z\M4\4810-M4Z\4810-M4Z-0223.wav
   wavs\Z\M4\4810-M4Z\4810-M4Z-0224.wav


In [46]:
import pandas as pd

metadata_source = pd.read_csv(
    "metadata.csv",
    sep="|",
    header=None,
    names=["path", "speaker_id", "text"],
    encoding="utf-8",
)

print(metadata_source.columns.tolist())
display(metadata_source.head())

['path', 'speaker_id', 'text']


,path,speaker_id,text
0,wavs/D/F2/0701-F2D/0701-F2D-0001.wav,7,džiugios naujienos iš čekijos
1,wavs/D/F2/0701-F2D/0701-F2D-0002.wav,7,sėkmė lydėjo mūsų mokyklos mokinį vilių
2,wavs/D/F2/0701-F2D/0701-F2D-0003.wav,7,čekijos sostinėje prahoje vykusiame tarptautin...
3,wavs/D/F2/0701-F2D/0701-F2D-0004.wav,7,festivalyje laimėta pirmoji vieta
4,wavs/D/F2/0701-F2D/0701-F2D-0005.wav,7,didžiuojamės ir džiaugiamės įspūdingu pasiekim...


In [47]:
selected_metadata = metadata_source[
    metadata_source["path"].str.match(
        r"wavs/Z/M4/9201-M4Z/9201-M4Z-\d+\.wav$"
    )
].copy()

print("Selected recordings:", len(selected_metadata))
display(selected_metadata.head())

Selected recordings: 327


,path,speaker_id,text
349369,wavs/Z/M4/9201-M4Z/9201-M4Z-0001.wav,92,antanas a jonynas
349370,wavs/Z/M4/9201-M4Z/9201-M4Z-0002.wav,92,apie hitu tapusį eilėraštį
349371,wavs/Z/M4/9201-M4Z/9201-M4Z-0003.wav,92,nežinau ar kalėdų eglutė apie kalėdas
349372,wavs/Z/M4/9201-M4Z/9201-M4Z-0004.wav,92,tūkstantis devyni šimtai aštuoniasdešimt septi...
349373,wavs/Z/M4/9201-M4Z/9201-M4Z-0005.wav,92,pirmą kartą nuskambėjusi grupės kardiofonas da...


In [ ]:
from pathlib import Path
import random
import shutil

SRC = Path("emo_extracted")
OUT = Path("dataset")

source_wavs = sorted(
    SRC.glob("wavs/Z/M4/9201-M4Z/*.wav")
)

wav_by_relative_path = {
    path.relative_to(SRC).as_posix(): path
    for path in source_wavs
}

rows = []

for _, row in selected_metadata.iterrows():
    relative_path = row["path"]
    source_path = wav_by_relative_path.get(relative_path)

    if source_path is None:
        print("Missing WAV:", relative_path)
        continue

    text = str(row["text"]).strip()

    if text and text.lower() != "nan":
        rows.append(
            {
                "source": source_path,
                "text": text.replace("|", " "),
            }
        )

print("Matched recordings:", len(rows))

Matched recordings: 327


In [49]:
random.Random(42).shuffle(rows)

validation_count = round(len(rows) * 0.10)
validation_rows = rows[:validation_count]
training_rows = rows[validation_count:]

(OUT / "wavs").mkdir(parents=True, exist_ok=True)

def create_split(rows, metadata_path, start_index):
    metadata_rows = []

    for index, item in enumerate(rows, start=start_index):
        filename = f"{index:04d}.wav"
        destination = OUT / "wavs" / filename

        shutil.copy2(item["source"], destination)

        metadata_rows.append(
            f"wavs/{filename}|{item['text']}"
        )

    metadata_path.write_text(
        "\n".join(metadata_rows) + "\n",
        encoding="utf-8",
    )

create_split(
    training_rows,
    OUT / "metadata_train.csv",
    start_index=1,
)

create_split(
    validation_rows,
    OUT / "metadata_val.csv",
    start_index=len(training_rows) + 1,
)

print("Training recordings:", len(training_rows))
print("Validation recordings:", len(validation_rows))

Training recordings: 294
Validation recordings: 33


In [50]:
recording_paths = sorted((OUT / "wavs").glob("*.wav"))
total_recordings = len(recording_paths)
total_seconds = sum(sf.info(path).duration for path in recording_paths)

hours, remainder = divmod(total_seconds, 3600)
minutes, seconds = divmod(remainder, 60)

print(f"Total recordings: {total_recordings}")
print(f"Total recording time: {int(hours)} h {int(minutes)} min {seconds:.2f} s")
print(f"Total recording time in seconds: {total_seconds:.2f}")

Total recordings: 327
Total recording time: 0 h 16 min 57.49 s
Total recording time in seconds: 1017.49


In [51]:
recording_paths = sorted((OUT / "wavs").glob("*.wav"))

if not recording_paths:
    print("No WAV recordings found.")
else:
    audio_parameters = [sf.info(path) for path in recording_paths]
    sampling_rates = sorted({info.samplerate for info in audio_parameters})
    bit_depths = sorted({info.subtype_info for info in audio_parameters})
    channel_counts = sorted({info.channels for info in audio_parameters})

    print("Main audio parameters:")
    print("Sampling frequency:", ", ".join(f"{rate} Hz" for rate in sampling_rates))
    print("Bit depth:", ", ".join(bit_depths))
    print("Number of channels:", ", ".join(map(str, channel_counts)))

Main audio parameters:
Sampling frequency: 96000 Hz
Bit depth: Signed 24 bit PCM
Number of channels: 1


In [ ]:
def split_statistics(metadata_path):
    metadata_rows = metadata_path.read_text(encoding="utf-8").splitlines()
    split_paths = [OUT / row.split("|", 1)[0] for row in metadata_rows if row.strip()]
    durations = [sf.info(path).duration for path in split_paths]
    return len(split_paths), sum(durations)


training_count, training_seconds = split_statistics(OUT / "metadata_train.csv")
validation_count, validation_seconds = split_statistics(OUT / "metadata_val.csv")

print("Training set:")
print(f"  Recordings: {training_count}")
print(f"  Total duration: {training_seconds / 3600:.2f} h ({training_seconds:.2f} s)")

print("Validation set:")
print(f"  Recordings: {validation_count}")
print(f"  Total duration: {validation_seconds / 3600:.2f} h ({validation_seconds:.2f} s)")

Training set:
  Recordings: 294
  Total duration: 0.25 h (912.70 s)
Validation set:
  Recordings: 33
  Total duration: 0.03 h (104.79 s)


In [ ]:
def durations_from_metadata(metadata_path):
    metadata_rows = metadata_path.read_text(encoding="utf-8").splitlines()
    paths = [
        OUT / row.split("|", 1)[0]
        for row in metadata_rows
        if row.strip()
    ]
    return [sf.info(path).duration for path in paths]


training_durations = durations_from_metadata(
    OUT / "metadata_train.csv"
)
validation_durations = durations_from_metadata(
    OUT / "metadata_val.csv"
)


for split_name, durations in [
    ("Training", training_durations),
    ("Validation", validation_durations),
]:
    if not durations:
        print(f"No {split_name.lower()} recording durations found.")
        continue

    print(f"{split_name} set:")
    print(f"  Minimum duration: {min(durations):.2f} s")
    print(f"  Maximum duration: {max(durations):.2f} s")
    print(f"  Average duration: {np.mean(durations):.2f} s")

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.hist(
        durations,
        bins=20,
        edgecolor="black",
    )

    ax.set_xlabel("Recording duration (s)")
    ax.set_ylabel("Number of recordings")
    ax.set_title(
        f"{split_name} recording duration distribution"
    )
    ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(
        f"{split_name.lower()}_recording_duration_histogram.png",
        dpi=150,
    )

    display(fig)
    plt.close(fig)

Training set:
  Minimum duration: 0.87 s
  Maximum duration: 7.89 s
  Average duration: 3.10 s


<Figure size 800x500 with 1 Axes>

Validation set:
  Minimum duration: 1.11 s
  Maximum duration: 7.38 s
  Average duration: 3.18 s


<Figure size 800x500 with 1 Axes>